In [76]:
import pandas as pd 
import numpy as np 
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# data loading

In [77]:
raw_df = pd.read_csv("../data/train_data/train_data.csv")

# EDA

In [78]:
raw_df.head(10)

,event_id,time_to_tca,mission_id,risk,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
0,0,1.566798,5,-10.204955,-7.834756,8.602101,14923.0,13792.0,453.8,5976.6,...,0.147350,58.272095,0.004092,0.165044,0.002987,0.386462,89.0,83.0,42.0,11.0
1,0,1.207494,5,-10.355758,-7.848937,8.956374,14544.0,13792.0,474.3,5821.2,...,0.059672,57.966413,0.003753,0.164383,0.002933,0.386393,89.0,83.0,42.0,11.0
2,0,0.952193,5,-10.345631,-7.847406,8.932195,14475.0,13792.0,474.6,5796.2,...,0.039258,57.907599,0.003576,0.164352,0.002967,0.386381,89.0,83.0,42.0,11.0
3,0,0.579669,5,-10.337809,-7.845880,8.913444,14579.0,13792.0,472.7,5838.9,...,0.022066,57.993905,0.003298,0.164309,0.002918,0.386400,89.0,83.0,40.0,14.0
4,0,0.257806,5,-10.391260,-7.852942,9.036838,14510.0,13792.0,478.7,5811.1,...,0.015075,57.946717,0.003670,0.164172,0.003220,0.386388,89.0,83.0,40.0,14.0
5,1,6.530455,5,-7.561299,-7.254301,2.746782,2392.0,3434.0,74.3,2317.1,...,0.278354,84.677411,0.006980,0.320622,0.004199,0.047385,71.0,88.0,0.0,2.0
6,1,5.561646,5,-9.315693,-7.468904,7.223137,3587.0,3434.0,99.0,3475.4,...,0.240907,63.860857,0.006402,0.264636,0.003725,0.040020,70.0,87.0,13.0,14.0
7,1,5.226504,5,-7.422508,-7.051001,2.956639,7882.0,3434.0,-50.0,-7638.3,...,0.240198,56.764910,0.005906,0.259109,0.003588,0.083247,70.0,87.0,13.0,14.0
8,1,3.570013,5,-9.248105,-7.327533,7.425994,26899.0,3434.0,-82.0,-26067.0,...,0.124802,30.242768,0.005883,0.174956,0.003408,0.058311,71.0,87.0,21.0,5.0
9,2,6.983474,2,-10.816161,-6.601713,13.293159,22902.0,14348.0,-1157.6,-6306.2,...,0.153332,39.695541,0.009370,0.269965,0.003886,0.339406,73.0,77.0,27.0,4.0


In [79]:
col_list = []
for column in raw_df.columns:
    col_list = np.append(col_list, column) 

print(col_list)

['event_id' 'time_to_tca' 'mission_id' 'risk' 'max_risk_estimate'
 'max_risk_scaling' 'miss_distance' 'relative_speed' 'relative_position_r'
 'relative_position_t' 'relative_position_n' 'relative_velocity_r'
 'relative_velocity_t' 'relative_velocity_n' 't_time_lastob_start'
 't_time_lastob_end' 't_recommended_od_span' 't_actual_od_span'
 't_obs_available' 't_obs_used' 't_residuals_accepted' 't_weighted_rms'
 't_rcs_estimate' 't_cd_area_over_mass' 't_cr_area_over_mass' 't_sedr'
 't_j2k_sma' 't_j2k_ecc' 't_j2k_inc' 't_ct_r' 't_cn_r' 't_cn_t'
 't_crdot_r' 't_crdot_t' 't_crdot_n' 't_ctdot_r' 't_ctdot_t' 't_ctdot_n'
 't_ctdot_rdot' 't_cndot_r' 't_cndot_t' 't_cndot_n' 't_cndot_rdot'
 't_cndot_tdot' 'c_object_type' 'c_time_lastob_start' 'c_time_lastob_end'
 'c_recommended_od_span' 'c_actual_od_span' 'c_obs_available' 'c_obs_used'
 'c_residuals_accepted' 'c_weighted_rms' 'c_rcs_estimate'
 'c_cd_area_over_mass' 'c_cr_area_over_mass' 'c_sedr' 'c_j2k_sma'
 'c_j2k_ecc' 'c_j2k_inc' 'c_ct_r' 'c_cn_r

looking at the data  
A single event_id might have multiple rows leading up to the exact moment they cross paths.

# feature engineering

In [80]:
# extracting the target feature 

df_sorted = raw_df.sort_values(by=['event_id', 'time_to_tca'], ascending=[True, False])

final_cdms = df_sorted.groupby('event_id').last().reset_index()

# creating the target labels
targets = final_cdms[['event_id', 'risk']].copy()
targets['is_hazard'] = (targets['risk'] >= -4.0).astype(int)

# Filter for all messages issued at least 2 days before the collision
df_early = df_sorted[df_sorted['time_to_tca'] >= 2.0]

features = df_early.groupby('event_id').last().reset_index()

master_df = pd.merge(features, targets[['event_id', 'is_hazard']], on='event_id')

In [81]:
display(master_df['is_hazard'].value_counts())

is_hazard
0    11931
1       11
Name: count, dtype: int64

thats an insane imbalance :)

after the sorting we must choose the columns we want in our data set and drop the rest to reduce the noise  
next is feature selection

In [82]:
# the model cant understand coordinate features like the position, velocity vectors and angles so...
# we will turn the vectors into distance and speed but "miss_distance", "relative_speed" already exist so the vectors will just get dropped

clean_df = master_df.drop(['relative_position_r',
                            'relative_position_t', 'relative_position_n', 'relative_velocity_r',
                            'relative_velocity_t', 'relative_velocity_n'], axis=1)

In [83]:
# the angles 'geocentric_latitude', 'azimuth', 'elevation' and 'x_j2k_inc' are turned into sin and cos

# clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']] = np.radians(clean_df[['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc']])
# clean_df[['geocentric_latitude_sin', 'azimuth_sin', 'elevation_sin', 't_j2k_inc_sin', 'c_j2k_inc_sin']] = np.sin(clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']])
# clean_df[['geocentric_latitude_cos', 'azimuth_cos', 'elevation_cos', 't_j2k_inc_cos', 'c_j2k_inc_cos']] = np.cos(clean_df[['geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad']])

# clean_df = clean_df.drop(['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc', 'geocentric_latitude_rad', 'azimuth_rad', 'elevation_rad', 't_j2k_inc_rad', 'c_j2k_inc_rad'], axis=1)

angles = ['geocentric_latitude', 'azimuth', 'elevation', 't_j2k_inc', 'c_j2k_inc']
for col in angles:
    clean_df[f'{col}_sin'] = np.sin(np.radians(clean_df[col]))
    clean_df[f'{col}_cos'] = np.cos(np.radians(clean_df[col]))
    clean_df = clean_df.drop(columns=[col])

In [84]:
# now looking at the data set most of the features left are noise and must be dropped 

# after sorting and extracting the target these features will cause data leakage if not dropped
leakage_features = ['event_id', 'mission_id', 'risk', 'max_risk_estimate', 'max_risk_scaling']
clean_df = clean_df.drop(leakage_features, axis= 1)      

# the model doesn't care about the probabilities of the radar so all the radar meta data will be dropped
meta_cols = [col for col in clean_df.columns if 'lastob' in col or 'obs_' in col or 'residuals' in col or 'od_span' in col or 'rms' in col]
clean_df = clean_df.drop(columns= meta_cols)

# now dropping all the cross sectional correlations 
cross_terms = [col for col in clean_df.columns if (col.startswith('t_c') or col.startswith('c_c')) and 'type' not in col and 'span' not in col and 'covariance' not in col]
clean_df = clean_df.drop(columns=cross_terms, errors='ignore')

# dropping the left un needed columns
clean_df = clean_df.drop(columns= ["time_to_tca"])

In [85]:
col_list = []
for column in clean_df.columns:
    col_list = np.append(col_list, column) 

print(col_list)

['miss_distance' 'relative_speed' 't_rcs_estimate' 't_sedr' 't_j2k_sma'
 't_j2k_ecc' 'c_object_type' 'c_rcs_estimate' 'c_sedr' 'c_j2k_sma'
 'c_j2k_ecc' 't_span' 'c_span' 't_h_apo' 't_h_per' 'c_h_apo' 'c_h_per'
 'mahalanobis_distance' 't_position_covariance_det'
 'c_position_covariance_det' 't_sigma_r' 'c_sigma_r' 't_sigma_t'
 'c_sigma_t' 't_sigma_n' 'c_sigma_n' 't_sigma_rdot' 'c_sigma_rdot'
 't_sigma_tdot' 'c_sigma_tdot' 't_sigma_ndot' 'c_sigma_ndot' 'F10' 'F3M'
 'SSN' 'AP' 'is_hazard' 'geocentric_latitude_sin'
 'geocentric_latitude_cos' 'azimuth_sin' 'azimuth_cos' 'elevation_sin'
 'elevation_cos' 't_j2k_inc_sin' 't_j2k_inc_cos' 'c_j2k_inc_sin'
 'c_j2k_inc_cos']


In [86]:
clean_df

,miss_distance,relative_speed,t_rcs_estimate,t_sedr,t_j2k_sma,t_j2k_ecc,c_object_type,c_rcs_estimate,c_sedr,c_j2k_sma,...,geocentric_latitude_sin,geocentric_latitude_cos,azimuth_sin,azimuth_cos,elevation_sin,elevation_cos,t_j2k_inc_sin,t_j2k_inc_cos,c_j2k_inc_sin,c_j2k_inc_cos
0,26899.0,3434.0,0.4030,0.000031,7001.561205,0.001028,DEBRIS,0.0045,0.008730,6880.588349,...,-0.810142,0.586234,-0.969054,0.246849,-0.016536,0.999863,0.990826,-0.135145,0.991288,0.131711
1,18763.0,14347.0,3.4479,0.000013,7158.408291,0.000863,UNKNOWN,NaN,0.000693,7168.395618,...,0.898052,0.439889,-0.275803,0.961214,-0.001004,0.999999,0.988956,-0.148208,0.938001,0.346633
2,23900.0,13574.0,0.4268,0.000019,7083.606025,0.002115,DEBRIS,0.0046,0.000487,7070.079861,...,-0.938993,0.343935,0.421301,0.906921,0.002925,0.999996,0.989897,-0.141789,0.944791,0.327675
3,33593.0,12093.0,NaN,0.000029,7082.429604,0.003942,DEBRIS,NaN,0.001403,7076.234143,...,-0.982538,0.186060,-0.587246,0.809409,-0.002133,0.999998,0.989386,-0.145308,0.988074,0.153979
4,304.0,2001.0,0.3690,0.000026,6995.466922,0.000732,PAYLOAD,0.3010,0.000039,6989.357563,...,0.990444,0.137912,0.990958,0.134172,-0.000250,1.000000,0.990765,-0.135589,0.990740,-0.135774
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11937,8239.0,12150.0,2.2340,0.000325,6821.521861,0.001275,ROCKET BODY,44.6054,0.007418,15207.145402,...,-0.117813,0.993036,0.780475,0.625187,-0.111839,0.993726,0.999001,0.044690,0.121008,0.992651
11938,3569.0,14952.0,13.3622,0.000018,6998.375714,0.002577,DEBRIS,0.0280,0.001267,6916.742616,...,0.752251,0.658877,-0.109811,0.993952,-0.001558,0.999999,0.990327,-0.138750,0.960993,0.276574
11939,34897.0,11876.0,0.4035,0.000019,6996.669840,0.000616,DEBRIS,0.0111,0.000965,6991.845532,...,-0.959676,0.281108,0.616497,0.787357,-0.002105,0.999998,0.990574,-0.136977,0.961256,0.275658
11940,24921.0,1156.0,2.5119,0.000011,7096.090754,0.001620,DEBRIS,0.0100,0.000138,7125.029171,...,0.605417,0.795908,-0.997974,0.063625,-0.114139,0.993465,0.999432,-0.033699,0.987980,-0.154582


# data cleaning

In [87]:
df_final = pd.get_dummies(clean_df, columns=['c_object_type'])

In [88]:
single_event = df_final[clean_df['is_hazard'] == 1] # Or whatever the first ID is
display(single_event)

,miss_distance,relative_speed,t_rcs_estimate,t_sedr,t_j2k_sma,t_j2k_ecc,c_rcs_estimate,c_sedr,c_j2k_sma,c_j2k_ecc,...,elevation_cos,t_j2k_inc_sin,t_j2k_inc_cos,c_j2k_inc_sin,c_j2k_inc_cos,c_object_type_DEBRIS,c_object_type_PAYLOAD,c_object_type_ROCKET BODY,c_object_type_TBA,c_object_type_UNKNOWN
1046,119.0,14985.0,0.2580,0.000050,6965.142968,0.002265,1.3221,0.000035,6966.938309,0.002519,...,0.999997,0.991060,-0.133418,0.990597,-0.136809,False,True,False,False,False
4710,352.0,14983.0,0.2296,0.000134,6961.649604,0.007014,4.3455,0.001725,6919.279083,0.000977,...,0.999997,0.991255,-0.131961,0.990348,-0.138601,True,False,False,False,False
5236,1827.0,212.0,15.8489,0.000015,7186.682782,0.000266,0.0038,0.000229,7176.644469,0.004716,...,0.986555,0.988534,-0.151000,0.988333,-0.152306,True,False,False,False,False
5744,492.0,4132.0,2.5119,0.000005,7090.488409,0.002705,NaN,0.000106,7040.224815,0.012023,...,0.999896,0.999425,-0.033917,0.960686,0.277637,False,False,False,False,True
7956,216.0,14940.0,2.8449,0.000029,7084.834132,0.000442,0.0072,0.001140,7077.027416,0.001336,...,1.000000,0.999345,-0.036180,0.997950,0.063999,True,False,False,False,False
8204,67.0,14768.0,4.5006,0.000016,7078.394938,0.000866,NaN,0.017858,8102.794932,0.181259,...,0.997470,0.990079,-0.140513,0.767402,0.641166,False,False,False,False,True
8484,3926.0,14562.0,15.8489,0.000007,7196.894857,0.001397,NaN,0.000221,7187.727455,0.001190,...,1.000000,0.988316,-0.152420,0.987185,-0.159581,False,False,False,False,True
8503,251.0,13342.0,3.4229,0.000010,7158.329778,0.002355,NaN,0.000430,7160.608470,0.004455,...,0.999997,0.988774,-0.149419,0.912732,0.408559,False,False,False,False,True
10143,9364.0,14724.0,9.2110,0.000111,7072.648404,0.001367,0.0050,0.010852,7052.836806,0.003025,...,1.000000,0.989611,-0.143769,0.987756,-0.156006,True,False,False,False,False
11355,473.0,14695.0,16.0709,0.000012,7201.476906,0.001246,0.0111,0.000141,7237.591733,0.005638,...,1.000000,0.988199,-0.153177,0.988614,-0.150472,True,False,False,False,False


4 of the 11 hazards has missing values so dropping the NA is not an option so we will instead use the median

In [89]:
X_train = df_final.drop(columns=['is_hazard'])
y_train = df_final['is_hazard']

In [90]:
X_train = X_train.fillna(X_train.median())

In [92]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

In [93]:
X_train_scaled

,miss_distance,relative_speed,t_rcs_estimate,t_sedr,t_j2k_sma,t_j2k_ecc,c_rcs_estimate,c_sedr,c_j2k_sma,c_j2k_ecc,...,elevation_cos,t_j2k_inc_sin,t_j2k_inc_cos,c_j2k_inc_sin,c_j2k_inc_cos,c_object_type_DEBRIS,c_object_type_PAYLOAD,c_object_type_ROCKET BODY,c_object_type_TBA,c_object_type_UNKNOWN
0,0.498632,-1.507290,-1.056997,-0.245238,-0.170523,-0.514183,-0.239531,0.109850,-0.241588,-0.044932,...,0.136748,-0.528462,-0.632702,0.309028,0.263329,0.907921,-0.358348,-0.115421,-0.034259,-0.691883
1,-0.012439,0.882565,0.201591,-0.269586,1.178625,-0.601828,-0.233875,-0.224663,-0.056419,-0.240833,...,0.149115,-0.936422,-0.800476,-0.196054,1.102033,-1.101418,-0.358348,-0.115421,-0.034259,1.445331
2,0.310247,0.713284,-1.047160,-0.261316,0.535200,0.061558,-0.239467,-0.233243,-0.119673,-0.201226,...,0.148773,-0.731156,-0.718033,-0.131697,1.028050,0.907921,-0.358348,-0.115421,-0.034259,-0.691883
3,0.919122,0.388958,-0.142021,-0.247685,0.525081,1.029232,-0.233875,-0.195125,-0.115713,-0.198940,...,0.148955,-0.842545,-0.763233,0.278564,0.350225,0.907921,-0.358348,-0.115421,-0.034259,-0.691883
4,-1.171958,-1.821105,-1.071051,-0.252208,-0.222945,-0.671112,-0.051104,-0.251917,-0.171608,-0.251025,...,0.149158,-0.541713,-0.638411,0.303830,-0.780494,-1.101418,2.790581,-0.115421,-0.034259,-0.691883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11937,-0.673513,0.401440,-0.300166,0.156017,-1.719165,-0.383626,28.104596,0.055260,5.115546,6.569231,...,-0.420416,1.255386,1.677016,-7.939870,3.623030,-1.101418,-0.358348,8.663931,-0.034259,-0.691883
11938,-0.966864,1.015054,4.299598,-0.263521,-0.197924,0.306350,-0.224597,-0.200788,-0.218327,-0.111303,...,0.149051,-0.637228,-0.679009,0.021872,0.828638,0.907921,-0.358348,-0.115421,-0.034259,-0.691883
11939,1.001033,0.341436,-1.056791,-0.261157,-0.212597,-0.732494,-0.235337,-0.213358,-0.170007,-0.242363,...,0.148960,-0.583363,-0.656231,0.024368,0.825062,0.907921,-0.358348,-0.115421,-0.034259,-0.691883
11940,0.374382,-2.006153,-0.185298,-0.271914,0.642590,-0.200542,-0.236036,-0.247783,-0.084320,-0.061061,...,-0.444164,1.349462,0.670215,0.277672,-0.853888,0.907921,-0.358348,-0.115421,-0.034259,-0.691883


In [95]:
print(f"Before SMOTE - Hazards: {sum(y_train==1)}, Safe: {sum(y_train==0)}")

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE  - Hazards: {sum(y_train_balanced==1)}, Safe: {sum(y_train_balanced==0)}")

Before SMOTE - Hazards: 11, Safe: 11931
After SMOTE  - Hazards: 11931, Safe: 11931
